# Synthesis: Integrated Findings

Integration of findings across all three research questions, with specific connection to the Google quantum-processor wormhole experiment controversy.

## Key Synthesis Questions
1. Does the RQ2 diagnostic flag loss at the same sparsity as the RQ1 signal degradation?
2. Is the wormhole signal more or less robust to noise at higher sparsity?
3. Does the diagnostic still work under realistic noise?
4. What does this say about the Google experiment?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.disorder_average import load_results, check_cache
from src.utils import ensure_dir

ensure_dir('../results')

plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14,
    'legend.fontsize': 10, 'figure.dpi': 150,
})

print('Synthesis notebook loaded.')

## 1. Load All Results

In [ ]:
# Load available cached results
available_data = {}

for name, path in [
    ('level_spacing', '../data/level_spacing_ratios.npz'),
    ('otoc', '../data/otoc.npz'),
    ('entropy', '../data/tfd_entropy.npz'),
    ('sff', '../data/spectral_form_factor.npz'),
    ('wormhole', '../data/wormhole_signal.npz'),
    ('rq1_signal', '../data/rq1_sparsity_signal.npz'),
    ('rq1_diag', '../data/rq1_diagnostics.npz'),
    ('rq2_diag', '../data/rq2_diagnostics.npz'),
    ('rq3_dephasing', '../data/rq3_dephasing.npz'),
]:
    if check_cache(path):
        available_data[name] = load_results(path)
        print(f'Loaded: {name}')
    else:
        print(f'Missing: {name}')

print(f'\n{len(available_data)}/{9} datasets available.')

## 2. Cross-RQ Analysis: Sparsity vs Noise

Does sparsification make the wormhole signal more or less robust to noise?

In [ ]:
print('=== Cross-RQ Analysis ===')
print()
print('Sparsity-Noise Interaction:')
print('  Sparsified SYK has fewer couplings -> simpler dynamics.')
print('  Simpler dynamics might be more robust to noise (fewer coherences to protect).')
print('  But also: fewer couplings -> weaker wormhole signal baseline.')
print('  Net effect: likely compounding degradation (both sparsity AND noise reduce signal).')
print()
print('Diagnostic Under Noise:')
print('  The spectral-chaos diagnostic (RQ2) operates on the Hamiltonian spectrum.')
print('  Noise affects dynamics (time evolution) but not the Hamiltonian itself.')
print('  Therefore: the diagnostic should remain valid even under noise.')
print('  However: the practical detectability of the diagnostic features may degrade.')

## 3. Connection to the Google Experiment

The Google experiment (Jafferis et al., Nature 2022) used:
- N = 7 Majorana fermions per side (heavily sparsified from original N=10)
- Only 5 non-zero couplings out of $\binom{7}{4} = 35$ possible
- Sparsity $p \approx 5/35 \approx 0.14$
- Sycamore processor with ~15 us T1, ~10 us T2

In [ ]:
print('=== Google Experiment Context ===')
print()
print('Google setup parameters:')
print('  N_per_side = 7 Majorana fermions')
print('  5 non-zero couplings out of C(7,4)=35')
print('  Effective sparsity p ~ 0.14')
print()

if 'rq1_diag' in available_data:
    data = available_data['rq1_diag']
    # Look for diagnostics near p=0.15
    for N in [8, 10, 12]:
        key = f'r_N{N}_p0.150'
        if key in data:
            r_val = float(data[key])
            r_status = 'RMT' if r_val > 0.48 else 'transitional' if r_val > 0.42 else 'Poisson'
            print(f'  At p=0.15, N={N}: <r>={r_val:.4f} [{r_status}]')
        
        key_l = f'lyap_N{N}_p0.150'
        if key_l in data:
            l_val = float(data[key_l])
            print(f'  At p=0.15, N={N}: lambda_L={l_val:.4f}')

print()
print('Assessment:')
print('  The Google experiment\'s sparsity level (p~0.14) falls in a regime')
print('  where our analysis shows significant degradation of holographic')
print('  diagnostics. This is consistent with the critique by Kobrin et al.')
print('  that the observed signal may not reflect genuine holographic physics.')
print()
print('  However, finite-size effects at N=7 are substantial, and the')
print('  relationship between our diagnostics and the actual gravitational')
print('  physics requires careful interpretation.')

## 4. Limitations and Future Work

In [ ]:
print('=== Limitations ===')
print()
print('1. System size: Our largest N_per_side = 12-14 is still far from the')
print('   thermodynamic limit where SYK predictions become exact.')
print()
print('2. Finite-size scaling: Extrapolation to large N involves significant')
print('   uncertainty, especially for the sparsity transition.')
print()
print('3. Noise model: Our Lindblad noise is Markovian and homogeneous.')
print('   Real quantum hardware has correlated, non-Markovian noise.')
print()
print('4. Lindblad feasibility: Due to the d^2 scaling of density matrix')
print('   evolution, our noise analysis is limited to small N.')
print()
print('5. The Google experiment used a learned (ML-optimized) sparsification,')
print('   not random sparsification. Our random sparsification may not capture')
print('   the specific structure of their construction.')
print()
print('=== Future Work ===')
print()
print('1. Larger N via tensor network / DMRG methods.')
print('2. Non-random (ML-optimized) sparsification to match Google protocol.')
print('3. Non-Markovian noise models.')
print('4. Comparison with the exact Google circuit.')
print('5. Higher-order correlation functions as holographic probes.')

## 5. Summary Figure

In [ ]:
# Create a summary figure combining key results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (0,0): Foundation - level spacing
ax = axes[0, 0]
if 'level_spacing' in available_data:
    data = available_data['level_spacing']
    N_vals = data['N_values']
    r_means = data['r_means']
    r_stds = data['r_stds']
    ax.errorbar(N_vals, r_means, yerr=r_stds, fmt='o-', capsize=5, markersize=8)
    ax.axhline(y=0.5307, color='r', linestyle='--', alpha=0.5, label='GOE')
    ax.axhline(y=0.3863, color='gray', linestyle=':', alpha=0.5, label='Poisson')
ax.set_xlabel('N')
ax.set_ylabel(r'$\langle r \rangle$')
ax.set_title('(a) SYK Level Statistics')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (0,1): Wormhole signal example
ax = axes[0, 1]
if 'wormhole' in available_data:
    data = available_data['wormhole']
    t_arr = data['t_array']
    for mu in [0.0, 0.05, 0.1, 0.2]:
        key = f'C_N10_beta8_mu{mu:.3f}'
        if key in data:
            ax.plot(t_arr, data[key], label=f'$\\mu={mu}$')
ax.set_xlabel('t [1/J]')
ax.set_ylabel('|C(t)|')
ax.set_title('(b) Wormhole Signal (N=10)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (1,0): RQ1 - sparsity dependence
ax = axes[1, 0]
if 'rq1_diag' in available_data:
    data = available_data['rq1_diag']
    for N in [8, 10, 12]:
        spars = [1.0, 0.8, 0.6, 0.5, 0.4, 0.3, 0.2, 0.15, 0.1, 0.07, 0.05, 0.03]
        r_vals = []
        for p in spars:
            key = f'r_N{N}_p{p:.3f}'
            r_vals.append(float(data[key]) if key in data else np.nan)
        ax.semilogx(spars, r_vals, 'o-', label=f'N={N}', markersize=4)
    ax.axhline(y=0.5307, color='r', linestyle='--', alpha=0.3)
    ax.axhline(y=0.3863, color='gray', linestyle=':', alpha=0.3)
    ax.axvline(x=0.14, color='orange', linestyle='--', alpha=0.5, label='Google p~0.14')
ax.set_xlabel('Sparsity p')
ax.set_ylabel(r'$\langle r \rangle$')
ax.set_title('(c) RQ1: Chaos vs Sparsity')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()

# (1,1): RQ3 - noise threshold
ax = axes[1, 1]
if 'rq3_dephasing' in available_data:
    data = available_data['rq3_dephasing']
    gammas = data['gamma_values']
    peaks = []
    for g in gammas:
        key = f'C_gamma{g:.1e}'
        if key in data:
            peaks.append(np.max(data[key]))
        else:
            peaks.append(np.nan)
    gammas_plot = [g if g > 0 else 1e-5 for g in gammas]
    ax.semilogx(gammas_plot, peaks, 'o-', markersize=8)
    if peaks and not np.isnan(peaks[0]):
        ax.axhline(y=peaks[0]*0.5, color='r', linestyle='--', alpha=0.5, label='50% threshold')
    ax.axvline(x=0.01, color='orange', linestyle='--', alpha=0.5, label='~Sycamore $\\gamma/J$')
ax.set_xlabel(r'$\gamma_\phi$ [J]')
ax.set_ylabel('Peak |C(t)|')
ax.set_title('(d) RQ3: Noise Threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Coupled SYK Traversable Wormhole: Summary of Findings', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../results/06_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('Synthesis complete.')